# Notebook 04 — Imputación, modelado y evaluación de precipitación diaria

**TFM:** *Evaluación de la influencia del componente espacial en la estimación de datos faltantes de precipitación diaria mediante modelos de Machine Learning aplicados a estaciones meteorológicas.*

## Objetivo

Este notebook se usan y comparan métodos tradicionales de imputación de registros de precipitación contra modelos de Machine Learning para el mismo fin, las evaluaciones se realizarán sobre **las mismas observaciones artificialmente ocultadas en el Notebook 03**.

Se comparan cuatro métodos divididos en dos pares:


**Tradicional**
---


1. **Media aritmética vecinal**
2. **IDW** — ponderación inversa de la distancia, con \(p=2\)

**Machine Learning**
---

3. **Random Forest base**
4. **Random Forest con representación espacial**, incorporando `P_IDW` como feature adicional

La comparación se estructura entre los pares con métodos equivalentes.

Primero se compara la media vecinal `(1)` con IDW `(2)` para evaluar el efecto de incorporar la distancia entre estaciones en un método tradicional.

La segunda comparación es entre Random Forest base `(3)` con un Random Forest que incorpora P_IDW como variable adicional `(4)`.

Las comparaciones se estructuran de esta manera con el fin de evaluar si una representación espacial explícita mejora el desempeño del modelo de Machine Learning. La interpretación se centrará en la variación de MAE y RMSE entre cada par de métodos.

## 1 - Librerías y rutas


---


El Notebook 03 ya realizó las comprobaciones del diseño experimental, la máscara artificial, los `NaN` naturales y la separación temporal, por lo que este notebook parte directamente desde estos archivos.

### 1.1 - Importación de librerías

In [ ]:
from pathlib import Path
import json

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error

### 1.2 - Importación de carpetas

In [ ]:
carpeta_actual = Path.cwd()

if carpeta_actual.name == 'notebooks':
    PROJECT_ROOT = carpeta_actual.parent
else:
    PROJECT_ROOT = carpeta_actual

folders = [
    PROJECT_ROOT / "data" / "raw"/ "precipitacion",
    PROJECT_ROOT / "data" / "intermediate",
    PROJECT_ROOT / "data" / "processed",
    PROJECT_ROOT / "notebooks"
]

for folder in folders:
    folder.mkdir(parents=True, exist_ok=True)

In [ ]:

RUTA_PROCESSED = PROJECT_ROOT / "data" / "processed"
RUTA_INTERMEDIATE = PROJECT_ROOT / "data" / "intermediate"
RUTA_TABLES = PROJECT_ROOT / "output" / "tables"
RUTA_FIGURES = PROJECT_ROOT / "output" / "figures"
RUTA_TABLES.mkdir(parents=True, exist_ok=True)
RUTA_FIGURES.mkdir(parents=True, exist_ok=True)

DATASET_PATH = RUTA_PROCESSED / "03_dataset_experimental.parquet"
Y_TRUE_PATH = RUTA_PROCESSED / "03_info_y_verdadera.parquet"
CONFIG_PATH = RUTA_PROCESSED / "03_config_experimento.json"
ESTACIONES_PATH = RUTA_INTERMEDIATE / "estaciones_seleccionadas.geojson"

## 2 - Lectura de las salidas del Notebook 03


---



In [ ]:
df = pd.read_parquet(DATASET_PATH)
info_y_verdadera = pd.read_parquet(Y_TRUE_PATH)

with open(CONFIG_PATH, "r", encoding="utf-8") as archivo:
    config = json.load(archivo)

ESTACION_OBJETIVO = str(config["estacion_objetivo"])
ESTACIONES_AUXILIARES = [str(x) for x in config["estaciones_auxiliares"]]
AUX1, AUX2 = ESTACIONES_AUXILIARES

print("Estación objetivo:", ESTACION_OBJETIVO)
print("Auxiliar 1:", AUX1)
print("Auxiliar 2:", AUX2)
print("Dataset experimental:", df.shape)
print("Observaciones con y_true:", info_y_verdadera.shape)



Estación objetivo: 21205710
Auxiliar 1: 21201230
Auxiliar 2: 21201600
Dataset experimental: (4383, 14)
Observaciones con y_true: (236, 5)


## 3 - Preparación de variables comunes

---




### 3.1 - Variables temporales

Se representa el día del año de forma cíclica mediante `doy_sin` y `doy_cos`.

Esto se hace mediante la transformación del día del año usando funciones seno y coseno para representar la estacionalidad anual de forma cíclica. De esta manera, fechas cercanas como el 31 de diciembre y el 1 de enero permanecen próximas en la representación utilizada por los modelos. Solo se hace con el Dataset con el que se va a entrenar `df`.

In [ ]:
df.index = pd.to_datetime(df.index)
info_y_verdadera.index = pd.to_datetime(info_y_verdadera.index)

df["doy"] = df.index.dayofyear # doy singnifica "Day Of the Year"
df["doy_sin"] = np.sin(2 * np.pi * df["doy"] / 365.25)
df["doy_cos"] = np.cos(2 * np.pi * df["doy"] / 365.25)

## 4 - Métodos de imputación de precipitación - Tradicionales


---


Se emplean dos métodos sencillos y directamente comparables:

1. **Media aritmética vecinal**, que asigna el mismo peso a las estaciones auxiliares disponibles.
2. **IDW**, que pondera las observaciones de acuerdo con su distancia a la estación objetivo.

Esta comparación permite evaluar directamente si incorporar la proximidad espacial mejora la estimación frente a un promedio no espacial.

### 4.1 - Media aritmética vecinal

Si existen dos auxiliares disponibles:

$$ \hat P_t = \frac{P_{1,t}+P_{2,t}}{2} $$

Si solamente una auxiliar dispone de información, se utiliza esa observación directamente.

In [ ]:
df["P_media_vecinal"] = df[[AUX1, AUX2]].mean(axis=1, skipna=True)

df

CodigoEstacion,21205710,21201230,21201600,es_nan_nat_obj,es_nan_nat_aux1,es_nan_nat_aux2,n_aux_data,Elegible,Completo_3_estaciones,split,es_lluvioso,mask_simulada,P_obj_original,P_obj_modelo,doy,doy_sin,doy_cos,P_media_vecinal
Fecha,,,,,,,,,,,,,,,,,,
2004-01-01,NaN,NaN,NaN,True,True,True,0,False,False,train,<NA>,False,NaN,NaN,1,0.017202,0.999852,NaN
2004-01-02,NaN,0.0,NaN,True,False,True,1,False,False,train,<NA>,False,NaN,NaN,2,0.034398,0.999408,0.00
2004-01-03,NaN,NaN,NaN,True,True,True,0,False,False,train,<NA>,False,NaN,NaN,3,0.051584,0.998669,NaN
2004-01-04,NaN,NaN,NaN,True,True,True,0,False,False,train,<NA>,False,NaN,NaN,4,0.068755,0.997634,NaN
2004-01-05,NaN,2.5,0.0,True,False,False,2,False,False,train,<NA>,False,NaN,NaN,5,0.085906,0.996303,1.25
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2015-12-27,NaN,0.0,NaN,True,False,True,1,False,False,test,<NA>,False,NaN,NaN,361,-0.073045,0.997329,0.00
2015-12-28,NaN,0.0,NaN,True,False,True,1,False,False,test,<NA>,False,NaN,NaN,362,-0.055879,0.998438,0.00
2015-12-29,NaN,0.0,NaN,True,False,True,1,False,False,test,<NA>,False,NaN,NaN,363,-0.038696,0.999251,0.00


### 4.2 - Distancias entre estaciones

Para IDW se calculan las distancias entre la estación objetivo y las auxiliares con la geometría Euclidiana.

$$ d = \sqrt{(x_2 - x_1)^2 + (y_2 - y_1)^2} $$

donde:
* $(x_1, y_1)$ son las coordenadas de la estación objetivo.
* $(x_2, y_2)$ son las coordenadas de la estación auxiliar.



In [ ]:
import geopandas as gpd

df_estaciones = gpd.read_file(ESTACIONES_PATH)
df_estaciones

,CODIGO,nombre,CATEGORIA,TECNOLOGIA,ESTADO,FECHA_INST,altitud,latitud,longitud,DEPARTAMEN,...,d_AREA_HID,d_ZONA_HID,d_CORRIENT,d_SUBZONA_,distancia_m,distancia_km,FECHA_INST_MOD,FECHA_SUSP_MOD,Objetivo,geometry
0,21205710,JARDIN BOTANICO [21205710],CP,"TEST001,TEST003",ESTA001,1974-09-15,2552.0,4.667867,-74.101034,11.0,...,Magdalena Cauca,Alto Magdalena,None,Río Bogotá,2709.989675,2.709990,1974-09-15,NaT,True,POINT (4877927.545 2073889.531)
1,21201230,ENMANUEL D' ALZON [21201230],PM,TEST003,ESTA002,1974-04-15,2520.0,4.701125,-74.070306,11.0,...,Magdalena Cauca,Alto Magdalena,None,Río Bogotá,4167.428588,4.167429,1974-04-15,2025-11-24,False,POINT (4881340.437 2077559.696)
2,21201600,SEDE IDEAM KRA 10 [21201600],PG,TEST003,ESTA002,1986-09-14,2685.0,4.607111,-74.072889,11.0,...,Magdalena Cauca,Alto Magdalena,None,Río Bogotá,6302.648333,6.302648,1986-09-14,2019-05-03,False,POINT (4881038.215 2067170.438)


In [ ]:
df_estaciones.crs

<Projected CRS: EPSG:9377>
Name: MAGNA-SIRGAS 2018 / Origen-Nacional
Axis Info [cartesian]:
- N[north]: Northing (metre)
- E[east]: Easting (metre)
Area of Use:
- name: Colombia - onshore and offshore. Includes San Andres y Providencia, Malpelo Islands, Roncador Bank, Serrana Bank and Serranilla Bank.
- bounds: (-84.77, -4.23, -66.87, 15.51)
Coordinate Operation:
- name: Colombia Transverse Mercator
- method: Transverse Mercator
Datum: Marco Geocentrico Nacional de Referencia 2018
- Ellipsoid: GRS 1980
- Prime Meridian: Greenwich

In [ ]:
# Geometría de la estación objetivo
geom_obj = df_estaciones.loc[
    df_estaciones["Objetivo"] == True,
    "geometry"
].iloc[0]

# Geometrías de las estaciones auxiliares

geom_aux1 = df_estaciones.loc[
    df_estaciones["CODIGO"] == AUX1,
    "geometry"
].iloc[0]

geom_aux2 = df_estaciones.loc[
    df_estaciones["CODIGO"] == AUX2,
    "geometry"
].iloc[0]

# Distancias en metros
DIST_AUX1_M = geom_obj.distance(geom_aux1)
DIST_AUX2_M = geom_obj.distance(geom_aux2)

# Convertir a kilómetros
DIST_AUX1_KM = DIST_AUX1_M / 1000
DIST_AUX2_KM = DIST_AUX2_M / 1000

print(f"Objetivo - Auxiliar 1: {DIST_AUX1_KM:.3f} km")
print(f"Objetivo - Auxiliar 2: {DIST_AUX2_KM:.3f} km")

Objetivo - Auxiliar 1: 5.012 km
Objetivo - Auxiliar 2: 7.404 km


### 4.3 - IDW — Inverse Distance Weighting

El método **Inverse Distance Weighting (IDW)** es una técnica de interpolación espacial, esta supone que las observaciones a menor distancia del punto de interés tienen una mayor influencia sobre el valor estimado, que aquellas más lejanas.

Para el TFM, IDW se utiliza para estimar la precipitación diaria en la estación objetivo a partir de las estaciones auxiliares disponibles. El aporte de cada estación auxiliar se pondera mediante su distancia a la estación objetivo:


$$ w_i = \frac{1}{d_i^p} $$


donde:

- ($w_i$) corresponde al peso asignado a la estación auxiliar ($i$).
- ($d_i$) es la distancia entre la estación auxiliar ($i$) y la estación objetivo.
- ($p$) es el exponente de ponderación.

Para este estudio se adopta:

$$p = 2$$

por lo que las estaciones más cercanas reciben un peso mayor en la estimación.

La precipitación estimada mediante IDW se calcula como:

$$P_{IDW}(t) =
\frac{\sum_i w_i P_i(t)}
{\sum_i w_i}$$

donde ($P_i(t)$) representa la precipitación observada en la estación auxiliar ($i$) para el día ($t$).

Cuando las estaciones auxiliares tienen una observación, la estimación es el promedio ponderado por distancia, caso contrario de solo tener una estación auxiliar con observación, se utiliza directamente su valor observado, al no haber otra estación disponible con la cual realizar el calculo.

A continuación, se determian los pesos para cada una de las estaciones.


In [ ]:

p = 2

peso_aux1 = 1 / DIST_AUX1_KM**p
peso_aux2 = 1 / DIST_AUX2_KM**p

print(f"El peso para la Estación auxiliar 1 ({AUX1}) es: {peso_aux1:.4f} ")
print(f"El peso para la Estación auxiliar 2 ({AUX2}) es: {peso_aux2:.4f} ")
print(f"El peso Total para las Estaciones es: {peso_aux2+peso_aux1:.4f} ")

El peso para la Estación auxiliar 1 (21201230) es: 0.0398 
El peso para la Estación auxiliar 2 (21201600) es: 0.0182 
El peso Total para las Estaciones es: 0.0581 


Los anteriores valores de los pesos correponden a valores sin normalizar, el peso que aparece en la fórmula no necesita sumar 1; la división final por la suma de los pesos es la que los normaliza automáticamente.


In [ ]:
peso_total = peso_aux1 + peso_aux2

print(f"Influencia relativa Auxiliar 1: {peso_aux1/peso_total*100:.1f}%")
print(f"Influencia relativa Auxiliar 2: {peso_aux2/peso_total*100:.1f}%")

Influencia relativa Auxiliar 1: 68.6%
Influencia relativa Auxiliar 2: 31.4%


Por lo anterior se puede determinar que la estación con mayor influencia es la estación 1 (21201230).

In [ ]:
def calcular_idw_fila(fila):

    valores = []
    pesos = []

    if pd.notna(fila[AUX1]):
        valores.append(fila[AUX1])
        pesos.append(peso_aux1)

    if pd.notna(fila[AUX2]):
        valores.append(fila[AUX2])
        pesos.append(peso_aux2)

    if len(valores) == 0:
        return np.nan

    return np.average(valores, weights=pesos)


df["P_IDW"] = df.apply(calcular_idw_fila,axis=1)

In [ ]:
df

CodigoEstacion,21205710,21201230,21201600,es_nan_nat_obj,es_nan_nat_aux1,es_nan_nat_aux2,n_aux_data,Elegible,Completo_3_estaciones,split,es_lluvioso,mask_simulada,P_obj_original,P_obj_modelo,doy,doy_sin,doy_cos,P_media_vecinal,P_IDW
Fecha,,,,,,,,,,,,,,,,,,,
2004-01-01,NaN,NaN,NaN,True,True,True,0,False,False,train,<NA>,False,NaN,NaN,1,0.017202,0.999852,NaN,NaN
2004-01-02,NaN,0.0,NaN,True,False,True,1,False,False,train,<NA>,False,NaN,NaN,2,0.034398,0.999408,0.00,0.000000
2004-01-03,NaN,NaN,NaN,True,True,True,0,False,False,train,<NA>,False,NaN,NaN,3,0.051584,0.998669,NaN,NaN
2004-01-04,NaN,NaN,NaN,True,True,True,0,False,False,train,<NA>,False,NaN,NaN,4,0.068755,0.997634,NaN,NaN
2004-01-05,NaN,2.5,0.0,True,False,False,2,False,False,train,<NA>,False,NaN,NaN,5,0.085906,0.996303,1.25,1.714479
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2015-12-27,NaN,0.0,NaN,True,False,True,1,False,False,test,<NA>,False,NaN,NaN,361,-0.073045,0.997329,0.00,0.000000
2015-12-28,NaN,0.0,NaN,True,False,True,1,False,False,test,<NA>,False,NaN,NaN,362,-0.055879,0.998438,0.00,0.000000
2015-12-29,NaN,0.0,NaN,True,False,True,1,False,False,test,<NA>,False,NaN,NaN,363,-0.038696,0.999251,0.00,0.000000


## 5 - Preparación del problema de Machine Learning


---


El modelo de ML (Random Forest - RF) base usa las precipitaciones auxiliares, indicadores de disponibilidad y estacionalidad. El RF con componente espacial añade la variable `P_IDW`, la cual representa la precipitación estimada mediante ponderación por distancia y permite evaluar si proporcionar esta información espacial adicional mejora el desempeño del modelo..

A continuación, se define qué columnas va a usar cada modelo como entrada y cuál es la variable que queremos predecir.

In [ ]:
FEATURES_BASE = [
    "P_aux1_rf",
    "P_aux2_rf",
    "aux1_disponible",
    "aux2_disponible",
    "doy_sin",
    "doy_cos"
]


Significa que el Random Forest base recibirá estas 6 variables:

 * `P_aux1_rf` =	precipitación de la auxiliar 1
 * `P_aux2_rf` =	precipitación de la auxiliar 2
 * `aux1_disponible` = indica si la auxiliar 1 tiene dato
 * `aux2_disponible` =	indica si la auxiliar 2 tiene dato
 * `doy_sin	posición` = temporal dentro del año
 * `doy_cos	posición` = temporal dentro del año

El Random Forest - Espacial recibirá la 6 variables anteriores, y adicional el `P_IDW` de la siguiente manera:

 * `P_aux1_rf` =	precipitación de la auxiliar 1

In [ ]:
FEATURES_SPATIAL = FEATURES_BASE + ["P_IDW"]

El target corresponde a la variable objetivo, la precipitación de la estación objetivo:

In [ ]:
TARGET = "P_obj_original"

In [ ]:
# Indicadores de disponibilidad de las estaciones auxiliares
df["aux1_disponible"] = df[AUX1].notna().astype(int)
df["aux2_disponible"] = df[AUX2].notna().astype(int)

# Variables de precipitación que recibirá Random Forest
# -1 representa técnicamente un dato faltante
df["P_aux1_rf"] = df[AUX1].fillna(-1)
df["P_aux2_rf"] = df[AUX2].fillna(-1)

mask_train = (df["split"] == "train") & df["Elegible"] & df[TARGET].notna()

X_train_base = df.loc[mask_train, FEATURES_BASE].copy()
X_train_spatial = df.loc[mask_train, FEATURES_SPATIAL].copy()
y_train = df.loc[mask_train, TARGET].copy()

print("Observaciones de entrenamiento:", len(y_train))

Observaciones de entrenamiento: 1657


## 6 - Funciones de evaluación



---



In [ ]:
def calcular_metricas(y_true, y_pred):
    return {
        "MAE": mean_absolute_error(y_true, y_pred),
        "RMSE": np.sqrt(mean_squared_error(y_true, y_pred))
    }

def obtener_fechas_evaluacion(split):
    return info_y_verdadera.index[info_y_verdadera["split"] == split]
    # recuperar las fechas artificialmente ocultadas correspondientes al conjunto solicitado (valid o test)

## 7 - Random Forest — selección sencilla mediante VALID

---

Se prueban pocas combinaciones razonables de hiperparámetros. La selección se realiza únicamente con `valid`, usando RMSE como criterio principal.

Se van a comprobar cuatro configuraciones diferentes de Random Forest

In [ ]:
PARAMETROS_RF = [
    {"n_estimators": 200, "max_depth": None, "min_samples_leaf": 1},
    {"n_estimators": 300, "max_depth": 10, "min_samples_leaf": 1},
    {"n_estimators": 300, "max_depth": 10, "min_samples_leaf": 2},
    {"n_estimators": 300, "max_depth": 6, "min_samples_leaf": 2},
]

In [ ]:
RANDOM_STATE_RF = 42

### 7.1 - Preparación de VALID

In [ ]:
fechas_valid = obtener_fechas_evaluacion("valid")
y_valid = info_y_verdadera.loc[fechas_valid, "y_true"]
X_valid_base = df.loc[fechas_valid, FEATURES_BASE]
X_valid_spatial = df.loc[fechas_valid, FEATURES_SPATIAL]
print("Observaciones de validación:", len(y_valid))

Observaciones de validación: 110


con lo anterior se extrajo:

* y_valid
→ valor real con el que comparar

* X_valid_base
→ información de entrada para RF base

* X_valid_spatial
→ información de entrada para RF espacial



### 7.2 - Ajuste del Random Forest base

In [ ]:
resultados_valid_base = []
for params in PARAMETROS_RF:
    modelo = RandomForestRegressor(**params, random_state=RANDOM_STATE_RF, n_jobs=-1)
    modelo.fit(X_train_base, y_train)
    pred = modelo.predict(X_valid_base)
    metricas = calcular_metricas(y_valid, pred)
    resultados_valid_base.append({**params, **metricas})

df_valid_base = pd.DataFrame(resultados_valid_base).sort_values("RMSE")
df_valid_base

,n_estimators,max_depth,min_samples_leaf,MAE,RMSE
3,300,6.0,2,3.517047,5.149325
2,300,10.0,2,3.695594,5.543840
1,300,10.0,1,3.832841,5.941785
0,200,NaN,1,3.891767,6.192389


In [ ]:
fila = df_valid_base.iloc[0]
MEJORES_PARAM_BASE = {
    "n_estimators": int(fila["n_estimators"]),
    "max_depth": None if pd.isna(fila["max_depth"]) else int(fila["max_depth"]),
    "min_samples_leaf": int(fila["min_samples_leaf"])
}
MEJORES_PARAM_BASE

{'n_estimators': 300, 'max_depth': 6, 'min_samples_leaf': 2}

Dentro de las configuraciones evaluadas para `RF base`, los siguientes parámetros presentaron menor error de validación:

* 300 árboles
* max_depth = 6
* min_samples_leaf = 2


### 7.3 - Ajuste del Random Forest Espacial

In [ ]:
resultados_valid_spatial = []
for params in PARAMETROS_RF:
    modelo = RandomForestRegressor(**params, random_state=RANDOM_STATE_RF, n_jobs=-1)
    modelo.fit(X_train_spatial, y_train)
    pred = modelo.predict(X_valid_spatial)
    metricas = calcular_metricas(y_valid, pred)
    resultados_valid_spatial.append({**params, **metricas})

df_valid_spatial = pd.DataFrame(resultados_valid_spatial).sort_values("RMSE")
df_valid_spatial

,n_estimators,max_depth,min_samples_leaf,MAE,RMSE
3,300,6.0,2,3.336075,4.935142
2,300,10.0,2,3.556331,5.337807
1,300,10.0,1,3.683920,5.668436
0,200,NaN,1,3.736057,5.837599


In [ ]:
fila = df_valid_spatial.iloc[0]
MEJORES_PARAM_SPATIAL = {
    "n_estimators": int(fila["n_estimators"]),
    "max_depth": None if pd.isna(fila["max_depth"]) else int(fila["max_depth"]),
    "min_samples_leaf": int(fila["min_samples_leaf"])
}
MEJORES_PARAM_SPATIAL

{'n_estimators': 300, 'max_depth': 6, 'min_samples_leaf': 2}

Dentro de las configuraciones evaluadas para `RF espacial`, los parámetros presentaron menor error de validación:

* 300 árboles
* max_depth = 6
* min_samples_leaf = 2

Cabe resaltar que coinciden con los mismos parámetros del modelo RF base, por lo que se puede hacer una comparación más limpia.

En el conjunto de validación, el Random Forest que incorpora la variable espacial derivada `P_IDW` presentó menores errores MAE y RMSE que el Random Forest base. La reducción fue aproximadamente del 5 % en MAE y del 4 % en RMSE.

## 8 - Entrenamiento de los modelos finales


---


En la etapa anterior se utilizaron las observaciones de `train` para entrenar distintas configuraciones de Random Forest y el conjunto `valid` para seleccionar los hiperparámetros con menor RMSE.

Una vez seleccionada la configuración de cada modelo, las observaciones de `train` y `valid` se combinan para realizar el entrenamiento definitivo.

Esta decisión permite aprovechar una mayor cantidad de información disponible antes de realizar la evaluación final, lo cual resulta especialmente relevante debido a la reducción del número de observaciones producida por los criterios de calidad y elegibilidad aplicados previamente.

El conjunto `test` permanece completamente separado y no participa ni en la selección de hiperparámetros ni en el entrenamiento final.

Por tanto, el flujo utilizado es:

- `train`: entrenamiento inicial de los modelos;
- `valid`: selección de hiperparámetros;
- `train + valid`: entrenamiento definitivo;
- `test`: evaluación final.

### 8.1 - Preparación de train + valid

In [ ]:
# Seleccionar las observaciones elegibles de train y valid
mask_train_valid = (
    df["split"].isin(["train", "valid"])
    & df["Elegible"]
    & df[TARGET].notna()
)

# Variables del Random Forest base
X_train_valid_base = df.loc[
    mask_train_valid,
    FEATURES_BASE
].copy()

# Variables del Random Forest espacial
X_train_valid_spatial = df.loc[
    mask_train_valid,
    FEATURES_SPATIAL
].copy()

# Valor real de precipitación de la estación objetivo
y_train_valid = df.loc[
    mask_train_valid,
    TARGET
].copy()

print(
    "Observaciones utilizadas en el entrenamiento final:",
    len(y_train_valid)
)

Observaciones utilizadas en el entrenamiento final: 2096


### 8.2 - Entrenamiento definitivo

Con los hiperparámetros seleccionados previamente, se construyen nuevamente los modelos Random Forest y se ajustan utilizando conjuntamente las observaciones de `train` y `valid`.

A partir de este punto no se realizan nuevas modificaciones de hiperparámetros. Los modelos obtenidos se consideran los modelos finales que serán evaluados sobre `test`.

In [ ]:
# Random Forest base definitivo
rf_base = RandomForestRegressor(
    **MEJORES_PARAM_BASE,
    random_state=RANDOM_STATE_RF,
    n_jobs=-1
)

# Random Forest espacial definitivo
rf_spatial = RandomForestRegressor(
    **MEJORES_PARAM_SPATIAL,
    random_state=RANDOM_STATE_RF,
    n_jobs=-1
)

En la celda anteior se contruyen los modelos con sus parámetros, a continuación se entrenan.

In [ ]:
rf_base.fit(
    X_train_valid_base,
    y_train_valid
)

rf_spatial.fit(
    X_train_valid_spatial,
    y_train_valid
)

print("✓ Modelos finales entrenados.")

✓ Modelos finales entrenados.


## 9 - Predicciones sobre TEST


---




Una vez entrenados los modelos definitivos, se generan las estimaciones para las observaciones artificialmente ocultadas pertenecientes exclusivamente al conjunto `test`.

Todos los métodos se evalúan sobre exactamente las mismas fechas, permitiendo una comparación directa de su desempeño.

Para cada observación se almacenan:

- el valor real conocido (`y_true`);
- la estimación mediante media vecinal;
- la estimación mediante IDW;
- la predicción del Random Forest base;
- la predicción del Random Forest espacial.

El conjunto `test` no ha participado previamente en el entrenamiento ni en la selección de hiperparámetros.

### 9.1 - Seleccionar TEST

In [ ]:
# Observaciones artificialmente ocultadas de TEST
datos_test = info_y_verdadera[
    info_y_verdadera["split"] == "test"
].copy()

# Fechas de evaluación
fechas_test = datos_test.index

# Tabla donde se almacenarán las predicciones
predicciones_test = datos_test[
    ["y_true", "n_aux_data"]
].copy()

print("Observaciones de test:", len(predicciones_test))


Observaciones de test: 126


### 9.2 - Métodos Tradicionales

In [ ]:
predicciones_test["Media_vecinal"] = df.loc[
    fechas_test,
    "P_media_vecinal"
]

predicciones_test["IDW"] = df.loc[
    fechas_test,
    "P_IDW"
]

### 9.3 - Random Forest Base

In [ ]:
X_test_base = df.loc[
    fechas_test,
    FEATURES_BASE
]

predicciones_test["RF_base"] = rf_base.predict(
    X_test_base
)

### 9.4 Random Forest Espacial

In [ ]:
X_test_spatial = df.loc[
    fechas_test,
    FEATURES_SPATIAL
]

predicciones_test["RF_espacial"] = rf_spatial.predict(
    X_test_spatial
)

In [ ]:
predicciones_test.head()

CodigoEstacion,y_true,n_aux_data,Media_vecinal,IDW,RF_base,RF_espacial
Fecha,,,,,,
2014-01-03,0.2,1,0.00,0.000000,3.269239,3.503127
2014-01-06,10.0,2,7.50,6.719675,6.483366,6.194414
2014-01-12,0.0,1,3.50,3.500000,1.168383,2.369721
2014-01-24,13.0,2,6.25,7.754912,7.684264,7.973926
2014-01-25,2.0,1,0.00,0.000000,3.353284,3.773599


## 10 - Evaluación final mediante MAE y RMSE


---



Las predicciones obtenidas sobre `test` se comparan con los valores reales almacenados en `y_true`.

Todos los métodos se evalúan utilizando exactamente las mismas observaciones, garantizando que las diferencias obtenidas correspondan al comportamiento de cada método y no a cambios en el conjunto de evaluación.

Se utilizan como métricas principales el Error Absoluto Medio (MAE) y la Raíz del Error Cuadrático Medio (RMSE).

### Error Absoluto Medio

$$
MAE =
\frac{1}{n}
\sum_{i=1}^{n}
|y_i-\hat{y}_i|
$$

El MAE representa, en milímetros, la diferencia absoluta promedio entre la precipitación real y la estimada.

### Raíz del Error Cuadrático Medio

$$
RMSE =
\sqrt{
\frac{1}{n}
\sum_{i=1}^{n}
(y_i-\hat{y}_i)^2
}
$$

El RMSE asigna mayor penalización a los errores grandes, por lo que complementa la información proporcionada por el MAE.

Las métricas obtenidas sobre `test` constituyen la evaluación final de los métodos.

### 10.1 - Cálculo de métricas

In [ ]:
modelos = [
    "Media_vecinal",
    "IDW",
    "RF_base",
    "RF_espacial"
]

resultados_test = []

for modelo in modelos:

    metricas = calcular_metricas(
        predicciones_test["y_true"],
        predicciones_test[modelo]
    )

    resultados_test.append({
        "Modelo": modelo,
        "n": len(predicciones_test),
        "MAE": metricas["MAE"],
        "RMSE": metricas["RMSE"]
    })

In [ ]:
metricas_test = pd.DataFrame(
    resultados_test
)

metricas_test

,Modelo,n,MAE,RMSE
0,Media_vecinal,126,2.178571,4.311971
1,IDW,126,2.217030,4.340116
2,RF_base,126,2.636259,4.039363
3,RF_espacial,126,2.761637,4.139828


In [ ]:
# Verificar que todos los métodos tengan predicciones
assert predicciones_test[["y_true"] + modelos].notna().all().all(), (
    "Existen valores faltantes en las predicciones de test."
)

## 11 Análisis de la influencia del Componente Espacial


---




Una vez obtenidas las métricas finales sobre `test`, se analiza específicamente el efecto producido por la incorporación de información espacial.

La evaluación se realiza mediante dos comparaciones:

### Métodos tradicionales

$$
\text{Media vecinal}
\leftrightarrow
\text{IDW}
$$

La media vecinal asigna el mismo peso a las estaciones auxiliares disponibles, mientras que IDW utiliza pesos derivados de la distancia a la estación objetivo.

Esta comparación permite analizar si la ponderación espacial mejora un procedimiento tradicional de estimación.

### Machine Learning

$$
RF_{base}
\leftrightarrow
RF_{espacial}
$$

Ambos Random Forest utilizan las mismas variables de precipitación auxiliar, disponibilidad y estacionalidad. El modelo espacial incorpora adicionalmente `P_IDW`, que representa una combinación de la precipitación auxiliar ponderada por la distancia entre estaciones.

De esta forma, la comparación permite analizar si proporcionar al modelo una representación espacial explícita produce una reducción del error de estimación.

In [ ]:
# Utilizar la tabla de métricas finales
resultados = metricas_test.set_index("Modelo")

# Media vecinal frente a IDW
cambio_mae_idw = cambio_porcentual(
    resultados.loc["Media_vecinal", "MAE"],
    resultados.loc["IDW", "MAE"]
)

cambio_rmse_idw = cambio_porcentual(
    resultados.loc["Media_vecinal", "RMSE"],
    resultados.loc["IDW", "RMSE"]
)

# Random Forest base frente a Random Forest espacial
cambio_mae_rf = cambio_porcentual(
    resultados.loc["RF_base", "MAE"],
    resultados.loc["RF_espacial", "MAE"]
)

cambio_rmse_rf = cambio_porcentual(
    resultados.loc["RF_base", "RMSE"],
    resultados.loc["RF_espacial", "RMSE"]
)

In [ ]:
print(f"Cambio MAE Media → IDW: {cambio_mae_idw:.2f}%")
print(f"Cambio RMSE Media → IDW: {cambio_rmse_idw:.2f}%")

print(f"Cambio MAE RF base → RF espacial: {cambio_mae_rf:.2f}%")
print(f"Cambio RMSE RF base → RF espacial: {cambio_rmse_rf:.2f}%")

Cambio MAE Media → IDW: 1.77%
Cambio RMSE Media → IDW: 0.65%
Cambio MAE RF base → RF espacial: 4.76%
Cambio RMSE RF base → RF espacial: 2.49%


### Interpretación del efecto espacial

Los cambios porcentuales obtenidos muestran que la incorporación explícita del componente espacial no produjo una reducción del error en el conjunto de prueba.

En los métodos tradicionales, el paso de la media vecinal a IDW produjo un aumento aproximado del **1.77 % en MAE** y del **0.65 % en RMSE**.

En los modelos de Machine Learning, la incorporación de `P_IDW` como variable espacial adicional produjo un aumento aproximado del **4.76 % en MAE** y del **2.49 % en RMSE** respecto al Random Forest base.

Por tanto, bajo la configuración experimental utilizada, la información espacial representada mediante ponderación por distancia no mejoró el desempeño de las estimaciones. Las diferencias observadas son relativamente pequeñas, por lo que el resultado debe interpretarse específicamente para esta combinación de estaciones, periodo de análisis y representación espacial empleada.